In [2]:
# # (optional) check if the installation is successful
# import torch_sparse
# import torch_geometric
# import torch_cluster
# import torch_scatter
# import torch_spline_conv

# Introduction

This notebook shows you how to run zero-shot with an ensemble of non-structure-informed models (ESM-1v, ESM-2 3B) with the ```zero_shot_esm_dms``` function as well as run zero-shot with a structure-informed model (ESM-IF) using ```zero_shot_esm_if_dms``` function. 

```zero_shot_esm_dms``` requires the wild-type amino acid sequence of the protein of the interest

```zero_shot_esm_if_dms``` requires the wild-type amino acid sequence and the structure of the protein of the interest, including the chain id of the protein of interest in the structure file.

Both functions return a dataframe with all the possible single amino acid mutations and their corresponding log likelihood ratio scores (i.e. the ratio of the likelihood compared to the wild-type sequence).

In [1]:
from Bio import SeqIO

from multievolve import zero_shot_esm_dms, zero_shot_esm_if_dms

/homes/xlian/.conda/envs/multievolve/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
wt_file = "../../data/dgoA/Q6BF16_V85A.fasta"
pdb_file = "/nfs/ml_lab/projects/ml_lab/xlian/dgoa_sim/boltz_results_Q6BF16_V85A/predictions/Q6BF16_V85A/Q6BF16_V85A_model_0.pdb"

In [5]:
wt_seq = str(SeqIO.read(wt_file, "fasta").seq)

esm_zeroshot = zero_shot_esm_dms(wt_seq)
esm_if_zeroshot = zero_shot_esm_if_dms(wt_seq, pdb_file, chain_id = 'A', scoring_strategy='wt-marginals')

/homes/xlian/.conda/envs/multievolve/lib/python3.11/site-packages/esm/pretrained.py:215: UserWarning: Regression weights not found, predicting contacts will not produce correct results.
  warnings.warn(


Native sequence from structure matches input sequence (205 residues)


The ```zero_shot_esm_dms``` returns a dataframe with the log likelihood ratio scores for each model as well as whether the mutation had a ratio greater than 1 indicated by the corresponding ```model#_pass``` column.

In [9]:
esm_zeroshot.head(10)
# most ">WT mutants come from F154 which should be ignored..."

,mutations,model_1_logratio,model_2_logratio,model_3_logratio,model_4_logratio,model_5_logratio,model_6_logratio,average_model_logratio,model_1_pass,model_2_pass,model_3_pass,model_4_pass,model_5_pass,model_6_pass,total_model_pass
2923,F154V,12.808342,12.932566,11.604060,11.827909,11.672048,9.404315,11.708206,1,1,1,1,1,1,6
1612,A85V,6.404394,6.751657,6.009520,4.950223,5.452200,7.721453,6.214908,1,1,1,1,1,1,6
2913,F154I,7.449529,3.854283,4.232207,3.873007,3.173041,2.359729,4.156966,1,1,1,1,1,1,6
1602,A85I,1.761732,1.927170,1.696910,0.126980,1.838047,3.704724,1.842594,1,1,1,1,1,1,6
2922,F154T,5.002196,8.293968,7.120539,4.434842,1.657275,-0.727311,4.296918,1,1,1,1,1,0,5
2907,F154A,2.087206,2.841445,2.408931,2.878347,0.911120,-0.968040,1.693168,1,1,1,1,1,0,5
2908,F154C,2.668057,2.909061,1.136784,2.387145,1.153551,-1.490005,1.460765,1,1,1,1,1,0,5
2915,F154L,4.526809,-1.164398,5.665892,1.256498,0.788321,-0.157212,1.819318,1,0,1,1,1,0,4
2921,F154S,1.632277,5.557922,1.797055,3.414243,-0.847916,-2.844104,1.451580,1,1,1,1,0,0,4
1722,S91P,0.762550,1.686811,1.920093,0.639447,-0.406222,-0.010005,0.765446,1,1,1,1,0,0,4


In [ ]:
# drop 14,37,85,154,126 positions in esm_zeroshot
excluded_positions = [14,37,85,154,126]

# how to filterout esm_zeroshot[mutations][-1:1] is not '154' liek this
# df['mutations'].apply(lambda x: int(x[1:-1]))
esm_zeroshot_filtered = esm_zeroshot[~esm_zeroshot['mutations'].apply(lambda x: int(x[1:-1]) in excluded_positions)]

# after filtering this positions out, result is expected as most positions only have 1 esm1v model pass...
esm_zeroshot_filtered.head(10)

,mutations,model_1_logratio,model_2_logratio,model_3_logratio,model_4_logratio,model_5_logratio,model_6_logratio,average_model_logratio,model_1_pass,model_2_pass,model_3_pass,model_4_pass,model_5_pass,model_6_pass,total_model_pass
1722,S91P,0.762550,1.686811,1.920093,0.639447,-0.406222,-0.010005,0.765446,1,1,1,1,0,0,4
1701,H90N,0.359705,-0.133402,-0.012197,0.647178,1.019192,0.620402,0.416813,1,0,0,1,1,1,4
2842,A150P,1.695085,0.876790,2.479300,-2.499054,0.609796,-4.160915,-0.166500,1,1,1,0,1,0,4
2567,Q136D,-2.441947,1.897441,-2.097108,0.540596,-0.778610,-7.836631,-1.786043,0,1,0,1,0,0,2
3154,D167A,-2.818139,-1.783514,0.223079,-0.779065,-1.286190,-5.459616,-1.983908,0,0,1,0,0,0,1
3161,D167K,-2.055824,-1.012349,0.227163,-1.467379,-0.218729,-7.890498,-2.069603,0,0,1,0,0,0,1
1693,H90D,-0.410871,0.398254,-4.044495,-2.606079,-3.212916,-3.395444,-2.211925,0,1,0,0,0,0,1
2570,Q136G,-2.888352,0.716825,-1.965678,-1.681326,-1.846018,-6.770049,-2.405766,0,1,0,0,0,0,1
522,V28L,0.438916,-4.256462,-1.350905,-4.315342,-2.825146,-5.259705,-2.928107,1,0,0,0,0,0,1
1865,G99E,0.992707,-3.060132,-3.943408,-3.261991,-1.996440,-10.822345,-3.681935,1,0,0,0,0,0,1


The ```zero_shot_esm_if_dms``` returns a dataframe with the log likelihood ratio scores.

In [7]:
esm_if_zeroshot.head(10)

,mutations,logratio
0,M1A,-1.732073
1,M1C,-4.675711
2,M1D,-3.391682
3,M1E,-3.743327
4,M1F,-5.301344
5,M1G,-0.260863
6,M1H,-2.604313
7,M1I,-5.016640
8,M1K,-3.883659
9,M1L,-4.309420
